# Sanity random ESM check

This notebook verifies on CPU that:
1. ESM scoring works.
2. ESM-filtered random mutation optimization runs end-to-end.

In [1]:
import os
import sys
import numpy as np

ROOT = os.path.abspath(os.path.join(os.getcwd()))
SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from pep_compass.models.esm.esm2_ppl import ESM2PPLScorer
from pep_compass.optimization.baselines.random_mutation import RandomMutationOptimizer
from pep_compass.optimization.black_box.apex_black_box import APEXBlackBox

print('Imports OK')

Imports OK


In [2]:
# 1) ESM CPU sanity check
sequence = 'KYCRRFRWLTFRWL'
scorer = ESM2PPLScorer(model_name='esm2_t6_8M_UR50D', device='cpu')
pll = scorer.pll(sequence)
passes = scorer.passes_threshold(sequence, threshold=-10.0)

assert np.isfinite(pll), f'PLL is not finite: {pll}'
assert isinstance(passes, bool), 'passes_threshold did not return bool'

print({'sequence': sequence, 'pll': pll, 'passes_at_-10': passes})

{'sequence': 'KYCRRFRWLTFRWL', 'pll': -0.686008095741272, 'passes_at_-10': True}


In [3]:
# 2) Full process sanity check: ESM-filtered random mutation with APEX black-box on CPU
bb = APEXBlackBox(
    mic_aggregate='mean',
    mic_bacteria=[1, 2, 3],
    device='cpu',
)

optimizer = RandomMutationOptimizer(
    black_box=bb,
    esm_model_name='esm2_t6_8M_UR50D',
    esm_ppl_threshold=-10.0,
    esm_device='cpu',
    esm_max_resampling_attempts=20,
)

result = optimizer.optimize(
    evaluation_budget=3,
    starting_point=sequence,
    rng_seed=123,
)

assert isinstance(result, dict), 'Result should be a dict'
assert 'best_x' in result and 'best_y' in result, f'Unexpected result keys: {result.keys()}'
print(result)
print('SANITY_OK')

/home/kjurasz/pep-compass/.venv/lib/python3.12/site-packages/poli_baselines/solvers/simple/random_mutation.py:39: UserWarning: In the initialization of the RandomMutation solver: 
The input is a 1D array, but no tokenizer was provided.
Assuming that the input is a string that can be tokenized
character by character using list(x_i).
  warnings.warn(


RandomMutationOptimizer result: {'best_x': 'KYCRRFRWLTFRWL', 'best_y': 2.879051446914673}
{'best_x': 'KYCRRFRWLTFRWL', 'best_y': 2.879051446914673}
SANITY_OK
